# Unified Benchmark — All Models on Full 63k Train

Trains LightGBM, XGBoost, CatBoost, Logistic Regression, and Torch MLP on
the **identical** 63,200-row PokerBench preflop train set, then evaluates
each on the 1,000-row test split with the standard 5-metric suite.

Results are persisted to `artifacts/unified_benchmark/results.json`.

In [ ]:
from __future__ import annotations

import json
import logging
import os
import sys
import time
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'poker_predictor').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')

ARTIFACTS_DIR = REPO_ROOT / 'artifacts' / 'unified_benchmark'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'artifacts -> {ARTIFACTS_DIR}')

In [ ]:
from poker_predictor.data.loaders import load_pokerbench_preflop
from poker_predictor.features.build import build_feature_matrix, canonical_action_label
from poker_predictor.training.labels import villain_fold_label

train_samples = load_pokerbench_preflop(split='train')
test_samples = load_pokerbench_preflop(split='test')
print(f'Train: {len(train_samples)} samples')
print(f'Test:  {len(test_samples)} samples')

In [ ]:
import numpy as np
import pandas as pd

X_train_full, raw_y_train = build_feature_matrix(train_samples)
X_test_full, raw_y_test = build_feature_matrix(test_samples)

y_train = [canonical_action_label(v) for v in raw_y_train]
y_test = [canonical_action_label(v) for v in raw_y_test]

mask_train = [v is not None for v in y_train]
mask_test = [v is not None for v in y_test]

X_train = X_train_full.loc[mask_train].reset_index(drop=True)
y_train_clean = [v for v in y_train if v is not None]
vy_train = [villain_fold_label(s) for s, m in zip(train_samples, mask_train) if m]

X_test = X_test_full.loc[mask_test].reset_index(drop=True)
y_test_clean = [v for v in y_test if v is not None]
vy_test = [villain_fold_label(s) for s, m in zip(test_samples, mask_test) if m]

print(f'Features: {X_train.shape[1]} cols')
print(f'Train: {len(X_train)} rows  |  Test: {len(X_test)} rows')

In [ ]:
from poker_predictor.models.baselines import MultiHeadModel, train_action_head, train_villain_fold_head
from poker_predictor.training.eval import evaluate
from sklearn.model_selection import train_test_split

MODELS = ['lightgbm', 'xgboost', 'catboost', 'logistic']
results = {}

for kind in MODELS:
    print(f'\n{"=" * 60}')
    print(f'Training: {kind}')
    print(f'{"=" * 60}')
    t0 = time.time()
    try:
        action_model, encoder = train_action_head(X_train, y_train_clean, kind=kind)
        villain_model = train_villain_fold_head(X_train, vy_train, kind=kind)
        fit_time = time.time() - t0

        model = MultiHeadModel(
            action_model=action_model,
            action_encoder=encoder,
            villain_fold_model=villain_model,
            feature_names=list(X_train.columns),
            meta={'model_kind': kind},
        )

        t1 = time.time()
        metrics = evaluate(model, X_test, y_test_clean, villain_y=vy_test, pot_bb=X_test['pot_bb'])
        pred_time = (time.time() - t1) * 1000

        metrics['fit_time_s'] = round(fit_time, 2)
        metrics['pred_time_ms'] = round(pred_time, 1)
        results[kind] = metrics

        model.save(ARTIFACTS_DIR / f'{kind}_multihead.joblib')
        print(f'  top1_accuracy: {metrics["top1_accuracy"]:.4f}')
        print(f'  fit: {fit_time:.1f}s  |  pred: {pred_time:.0f}ms')
    except Exception as e:
        print(f'  FAILED: {e}')
        results[kind] = {'error': str(e)}

In [ ]:
from poker_predictor.training.train_torch import train_torch

print(f'\n{"=" * 60}')
print('Training: torch_mlp')
print(f'{"=" * 60}')
t0 = time.time()
torch_result = train_torch(X_train_full.loc[mask_train].reset_index(drop=True),
                           raw_y_train, vy_train, output_dir=ARTIFACTS_DIR)
torch_fit_time = time.time() - t0
results['torch_mlp'] = {
    'val_acc': torch_result['val_acc'],
    'fit_time_s': round(torch_fit_time, 2),
    'note': 'val_acc from internal 90/10 split; no full test eval (not a MultiHeadModel)',
}
print(f'  val_acc: {torch_result["val_acc"]:.4f}')
print(f'  fit: {torch_fit_time:.1f}s')

In [ ]:
import pandas as pd

df = pd.DataFrame(results).T
cols = ['top1_accuracy', 'action_log_loss', 'villain_fold_brier',
        'bluff_ev_mean', 'bluff_positive_frac', 'fit_time_s', 'pred_time_ms']
display_cols = [c for c in cols if c in df.columns]
df[display_cols].sort_values('top1_accuracy', ascending=False)

In [ ]:
out_path = ARTIFACTS_DIR / 'results.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f'Saved results to {out_path}')

## Summary

All models trained on identical 63k data, evaluated on 1k test split.
LightGBM remains the strongest with the multi-head architecture;
XGBoost is competitive; CatBoost and Logistic provide useful diversity
for ensembling.